# Document-Level BM25 Retriever for Wikipedia Pets

This notebook builds a document-level lexical retrieval pipeline for the Wikipedia pets corpus.

- BM25 is a lexical retrieval model.
- This notebook indexes whole documents, not chunks.
- It complements the MiniLM/SBERT semantic retriever instead of replacing it.
- BM25 is especially useful for exact term matching, rare terms, breed names, scientific names, and terminology-heavy queries.
- A later hybrid system can combine BM25 and MiniLM with Reciprocal Rank Fusion or another document-level fusion strategy.

## Dependency Notes

Required packages for this notebook:

- `pandas`
- `numpy`
- `pyarrow`
- `rank-bm25`
- `tqdm`

This notebook uses a built-in stopword list, so `nltk` and `scikit-learn` are not required.

Install command:

```bash
pip install pandas numpy pyarrow rank-bm25 tqdm
```

In [2]:
from pathlib import Path
from collections import Counter
import gc
import json
import pickle
import re
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

d:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
REPO_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(
    (
        path
        for path in REPO_ROOT_CANDIDATES
        if (path / 'data').exists() and (path / 'output').exists()
    ),
    Path.cwd().resolve(),
)

CORPUS_DIR = REPO_ROOT / 'data' / 'corpus_cleaned'
PAGE_INDEX_FILE = REPO_ROOT / 'output' / 'wikipedia-pets' / 'page_index.json'
BM25_DIR = REPO_ROOT / 'data' / 'bm25_index'
DOCUMENT_METADATA_FILE = BM25_DIR / 'document_metadata.parquet'
TOKENIZED_CORPUS_FILE = BM25_DIR / 'tokenized_corpus.pkl'
BM25_INDEX_FILE = BM25_DIR / 'bm25_index.pkl'
BM25_CONFIG_FILE = BM25_DIR / 'bm25_config.json'
SKIPPED_DOCUMENTS_FILE = BM25_DIR / 'skipped_documents.parquet'
SEARCH_DEMO_RESULTS_FILE = BM25_DIR / 'search_demo_results.parquet'

BM25_DIR.mkdir(parents=True, exist_ok=True)

MIN_TOKEN_LENGTH = 2
LOWERCASE = True
REMOVE_STOPWORDS = True
KEEP_NUMBERS = True
BM25_K1 = 1.5
BM25_B = 0.75

STOPWORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and', 'any', 'are', 'as',
    'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can',
    'could', 'did', 'do', 'does', 'doing', 'down', 'during', 'each', 'few', 'for', 'from', 'further',
    'had', 'has', 'have', 'having', 'he', 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his',
    'how', 'i', 'if', 'in', 'into', 'is', 'it', 'its', 'itself', 'just', 'me', 'more', 'most', 'my',
    'myself', 'no', 'nor', 'not', 'now', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our',
    'ours', 'ourselves', 'out', 'over', 'own', 'same', 'she', 'should', 'so', 'some', 'such', 'than',
    'that', 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these', 'they', 'this',
    'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 'was', 'we', 'were', 'what',
    'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will', 'with', 'you', 'your', 'yours',
    'yourself', 'yourselves'
}

print(f'Repo root: {REPO_ROOT}')
print(f'Corpus dir: {CORPUS_DIR}')
print(f'Page index file: {PAGE_INDEX_FILE}')
print(f'BM25 dir: {BM25_DIR}')

Repo root: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization
Corpus dir: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\corpus_cleaned
Page index file: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\output\wikipedia-pets\page_index.json
BM25 dir: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\bm25_index


In [4]:
def utc_timestamp():
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())


def read_file_contents(file_path):
    file_path = Path(file_path)
    with file_path.open('r', encoding='utf-8-sig', errors='replace') as handle:
        return handle.read()


def normalize_text(text):
    text = str(text)
    if LOWERCASE:
        text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize_for_bm25(text):
    normalized = normalize_text(text)
    token_pattern = r"[a-z]+(?:['’][a-z]+)*|\d+(?:\.\d+)*" if KEEP_NUMBERS else r"[a-z]+(?:['’][a-z]+)*"
    raw_tokens = re.findall(token_pattern, normalized)

    tokens = []
    for token in raw_tokens:
        token = token.replace("'", '').replace('’', '')
        token = token.strip()
        if len(token) < MIN_TOKEN_LENGTH:
            continue
        if REMOVE_STOPWORDS and token in STOPWORDS:
            continue
        tokens.append(token)

    return tokens

In [5]:
def build_document_metadata(overwrite=False):
    if DOCUMENT_METADATA_FILE.exists() and not overwrite:
        print(f'Loading existing document metadata from {DOCUMENT_METADATA_FILE}')
        return pd.read_parquet(DOCUMENT_METADATA_FILE).sort_values('doc_id').reset_index(drop=True)

    if not PAGE_INDEX_FILE.exists():
        raise FileNotFoundError(f'Missing page index file: {PAGE_INDEX_FILE}')
    if not CORPUS_DIR.exists():
        raise FileNotFoundError(f'Missing corpus directory: {CORPUS_DIR}')

    page_index = json.loads(read_file_contents(PAGE_INDEX_FILE))
    documents = []
    skipped_documents = []
    start_time = time.time()

    for entry_index, entry in enumerate(page_index, start=1):
        page_id = entry.get('page_id')
        title = entry.get('title')
        text_file_original = entry.get('text_file')

        if not text_file_original:
            skipped_documents.append(
                {
                    'page_id': page_id,
                    'title': title,
                    'file_name': None,
                    'reason': 'missing text_file',
                }
            )
        else:
            file_name = text_file_original[6:] if str(text_file_original).startswith('texts/') else str(text_file_original)
            document_path = CORPUS_DIR / file_name

            if not document_path.exists():
                skipped_documents.append(
                    {
                        'page_id': page_id,
                        'title': title,
                        'file_name': file_name,
                        'reason': 'missing corpus file',
                    }
                )
            else:
                document_text = read_file_contents(document_path).strip()
                if not document_text:
                    skipped_documents.append(
                        {
                            'page_id': page_id,
                            'title': title,
                            'file_name': file_name,
                            'reason': 'empty document text',
                        }
                    )
                else:
                    documents.append(
                        {
                            'doc_id': len(documents),
                            'page_id': page_id,
                            'title': title,
                            'file_name': file_name,
                            'text_file_original': text_file_original,
                            'word_count': len(document_text.split()),
                            'char_count': len(document_text),
                            'document_text': document_text,
                        }
                    )

        if entry_index % 500 == 0 or entry_index == len(page_index):
            elapsed = time.time() - start_time
            print(
                f'Processed {entry_index:,}/{len(page_index):,} page index entries in {elapsed:.1f}s '
                f'| documents: {len(documents):,} | skipped: {len(skipped_documents):,}'
            )

    document_df = pd.DataFrame(
        documents,
        columns=[
            'doc_id',
            'page_id',
            'title',
            'file_name',
            'text_file_original',
            'word_count',
            'char_count',
            'document_text',
        ],
    )
    skipped_df = pd.DataFrame(
        skipped_documents,
        columns=['page_id', 'title', 'file_name', 'reason'],
    )

    document_df.to_parquet(DOCUMENT_METADATA_FILE, index=False)
    skipped_df.to_parquet(SKIPPED_DOCUMENTS_FILE, index=False)

    print(f'Saved document metadata to {DOCUMENT_METADATA_FILE}')
    print(f'Saved skipped documents to {SKIPPED_DOCUMENTS_FILE}')
    print(f'Final document count: {len(document_df):,}')
    print(f'Final skipped count: {len(skipped_df):,}')

    return document_df

In [6]:
def load_document_metadata():
    if not DOCUMENT_METADATA_FILE.exists():
        raise FileNotFoundError(f'Missing document metadata file: {DOCUMENT_METADATA_FILE}')
    return pd.read_parquet(DOCUMENT_METADATA_FILE).sort_values('doc_id').reset_index(drop=True)


def load_tokenized_corpus():
    if not TOKENIZED_CORPUS_FILE.exists():
        raise FileNotFoundError(f'Missing tokenized corpus file: {TOKENIZED_CORPUS_FILE}')
    with TOKENIZED_CORPUS_FILE.open('rb') as handle:
        return pickle.load(handle)


def load_bm25_index():
    if not BM25_INDEX_FILE.exists():
        raise FileNotFoundError(f'Missing BM25 index file: {BM25_INDEX_FILE}')
    with BM25_INDEX_FILE.open('rb') as handle:
        return pickle.load(handle)


def _tokenized_corpus_stats(tokenized_corpus):
    document_lengths = np.array([len(tokens) for tokens in tokenized_corpus], dtype=np.int32)
    empty_token_documents = int((document_lengths == 0).sum())

    return {
        'document_count': int(len(tokenized_corpus)),
        'total_tokens': int(document_lengths.sum()) if len(document_lengths) else 0,
        'average_document_token_count': float(document_lengths.mean()) if len(document_lengths) else 0.0,
        'median_document_token_count': float(np.median(document_lengths)) if len(document_lengths) else 0.0,
        'min_document_token_count': int(document_lengths.min()) if len(document_lengths) else 0,
        'max_document_token_count': int(document_lengths.max()) if len(document_lengths) else 0,
        'empty_token_documents': empty_token_documents,
    }


def _print_tokenized_corpus_stats(stats):
    print('Tokenized corpus statistics')
    print(f"- documents: {stats['document_count']:,}")
    print(f"- total tokens: {stats['total_tokens']:,}")
    print(f"- average document token count: {stats['average_document_token_count']:.2f}")
    print(f"- median document token count: {stats['median_document_token_count']:.2f}")
    print(f"- min document token count: {stats['min_document_token_count']}")
    print(f"- max document token count: {stats['max_document_token_count']}")
    print(f"- empty-token documents: {stats['empty_token_documents']}")


def build_tokenized_corpus(overwrite=False):
    if TOKENIZED_CORPUS_FILE.exists() and not overwrite:
        print(f'Loading existing tokenized corpus from {TOKENIZED_CORPUS_FILE}')
        tokenized_corpus = load_tokenized_corpus()
        _print_tokenized_corpus_stats(_tokenized_corpus_stats(tokenized_corpus))
        return tokenized_corpus

    document_metadata = load_document_metadata()
    document_metadata = document_metadata.sort_values('doc_id').reset_index(drop=True)

    tokenized_corpus = []
    for text in tqdm(document_metadata['document_text'], desc='Tokenizing documents'):
        tokenized_corpus.append(tokenize_for_bm25(text))

    with TOKENIZED_CORPUS_FILE.open('wb') as handle:
        pickle.dump(tokenized_corpus, handle, protocol=pickle.HIGHEST_PROTOCOL)

    stats = _tokenized_corpus_stats(tokenized_corpus)
    _print_tokenized_corpus_stats(stats)
    print(f'Saved tokenized corpus to {TOKENIZED_CORPUS_FILE}')

    del document_metadata
    gc.collect()
    return tokenized_corpus


def build_bm25_index(overwrite=False):
    if BM25_INDEX_FILE.exists() and not overwrite:
        print(f'Loading existing BM25 index from {BM25_INDEX_FILE}')
        return load_bm25_index()

    tokenized_corpus = load_tokenized_corpus()
    build_start = time.time()
    bm25 = BM25Okapi(tokenized_corpus, k1=BM25_K1, b=BM25_B)

    with BM25_INDEX_FILE.open('wb') as handle:
        pickle.dump(bm25, handle, protocol=pickle.HIGHEST_PROTOCOL)

    config = {
        'model': 'BM25Okapi',
        'k1': BM25_K1,
        'b': BM25_B,
        'min_token_length': MIN_TOKEN_LENGTH,
        'lowercase': LOWERCASE,
        'remove_stopwords': REMOVE_STOPWORDS,
        'keep_numbers': KEEP_NUMBERS,
        'document_metadata_file': str(DOCUMENT_METADATA_FILE),
        'tokenized_corpus_file': str(TOKENIZED_CORPUS_FILE),
        'bm25_index_file': str(BM25_INDEX_FILE),
        'created_at': utc_timestamp(),
    }
    BM25_CONFIG_FILE.write_text(json.dumps(config, indent=2), encoding='utf-8')

    elapsed = time.time() - build_start
    print(f'Saved BM25 index to {BM25_INDEX_FILE}')
    print(f'Saved BM25 config to {BM25_CONFIG_FILE}')
    print(f'Built BM25 index in {elapsed:.1f}s')

    gc.collect()
    return bm25

In [7]:
def _empty_bm25_results(include_document_text=True):
    columns = [
        'rank',
        'bm25_score',
        'doc_id',
        'page_id',
        'title',
        'file_name',
        'word_count',
        'char_count',
        'matched_query_tokens',
    ]
    if include_document_text:
        columns.append('document_text')
    return pd.DataFrame(columns=columns)


def bm25_search(query, top_k=10, include_document_text=True):
    if top_k <= 0:
        raise ValueError('top_k must be a positive integer')

    bm25 = load_bm25_index()
    document_metadata = load_document_metadata()
    tokenized_corpus = load_tokenized_corpus()

    if len(document_metadata) != len(tokenized_corpus):
        raise ValueError('Document metadata row count does not match tokenized corpus length.')

    query_tokens = tokenize_for_bm25(query)
    if not query_tokens:
        return _empty_bm25_results(include_document_text=include_document_text)

    scores = np.asarray(bm25.get_scores(query_tokens), dtype=np.float32)
    if scores.size == 0:
        return _empty_bm25_results(include_document_text=include_document_text)

    top_k = min(top_k, scores.shape[0])
    if top_k == scores.shape[0]:
        candidate_indices = np.arange(scores.shape[0])
    else:
        candidate_indices = np.argpartition(scores, -top_k)[-top_k:]

    ranked_indices = candidate_indices[np.argsort(scores[candidate_indices])[::-1]]

    results = []
    unique_query_tokens = list(dict.fromkeys(query_tokens))
    for rank, doc_id in enumerate(ranked_indices, start=1):
        doc_row = document_metadata.iloc[int(doc_id)]
        document_token_set = set(tokenized_corpus[int(doc_id)])
        matched_query_tokens = [token for token in unique_query_tokens if token in document_token_set]

        result = {
            'rank': rank,
            'bm25_score': float(scores[int(doc_id)]),
            'doc_id': int(doc_row['doc_id']),
            'page_id': int(doc_row['page_id']) if pd.notna(doc_row['page_id']) else np.nan,
            'title': doc_row['title'],
            'file_name': doc_row['file_name'],
            'word_count': int(doc_row['word_count']),
            'char_count': int(doc_row['char_count']),
            'matched_query_tokens': ', '.join(matched_query_tokens),
        }
        if include_document_text:
            result['document_text'] = doc_row['document_text']

        results.append(result)

    return pd.DataFrame(results)


def explain_bm25_result(query, doc_id):
    document_metadata = load_document_metadata()
    tokenized_corpus = load_tokenized_corpus()

    if doc_id < 0 or doc_id >= len(document_metadata):
        raise ValueError(f'doc_id must be between 0 and {len(document_metadata) - 1}, got {doc_id}')

    query_tokens = tokenize_for_bm25(query)
    document_row = document_metadata.iloc[int(doc_id)]
    document_tokens = tokenized_corpus[int(doc_id)]
    token_counts = Counter(document_tokens)
    matched_query_tokens = [token for token in list(dict.fromkeys(query_tokens)) if token in token_counts]
    term_frequencies = {token: int(token_counts[token]) for token in matched_query_tokens}

    summary = {
        'document_title': document_row['title'],
        'page_id': int(document_row['page_id']) if pd.notna(document_row['page_id']) else np.nan,
        'file_name': document_row['file_name'],
        'query_tokens': query_tokens,
        'matched_query_tokens': matched_query_tokens,
        'term_frequencies': term_frequencies,
        'document_token_count': int(len(document_tokens)),
    }

    print(f"Document title: {summary['document_title']}")
    print(f"Page ID: {summary['page_id']}")
    print(f"File name: {summary['file_name']}")
    print(f"Query tokens: {summary['query_tokens']}")
    print(f"Matched query tokens: {summary['matched_query_tokens']}")
    print(f"Term frequencies: {summary['term_frequencies']}")
    print(f"Document token count: {summary['document_token_count']}")

    return summary


def reciprocal_rank_fusion(bm25_results, semantic_results, k=60):
    if k <= 0:
        raise ValueError('k must be a positive integer')

    required_columns = {'doc_id', 'rank'}
    if not required_columns.issubset(bm25_results.columns):
        raise ValueError('bm25_results must contain doc_id and rank columns')
    if not required_columns.issubset(semantic_results.columns):
        raise ValueError('semantic_results must contain doc_id and rank columns')

    bm25_rank_df = bm25_results[['doc_id', 'rank']].rename(columns={'rank': 'bm25_rank'}).copy()
    semantic_rank_df = semantic_results[['doc_id', 'rank']].rename(columns={'rank': 'semantic_rank'}).copy()

    fused = bm25_rank_df.merge(semantic_rank_df, on='doc_id', how='outer')
    fused['rrf_score'] = 0.0

    bm25_mask = fused['bm25_rank'].notna()
    semantic_mask = fused['semantic_rank'].notna()
    fused.loc[bm25_mask, 'rrf_score'] += 1.0 / (k + fused.loc[bm25_mask, 'bm25_rank'])
    fused.loc[semantic_mask, 'rrf_score'] += 1.0 / (k + fused.loc[semantic_mask, 'semantic_rank'])

    return fused.sort_values(['rrf_score', 'doc_id'], ascending=[False, True]).reset_index(drop=True)

In [12]:
def validate_bm25_pipeline():
    if not DOCUMENT_METADATA_FILE.exists():
        raise FileNotFoundError(f'Missing document metadata file: {DOCUMENT_METADATA_FILE}')
    if not TOKENIZED_CORPUS_FILE.exists():
        raise FileNotFoundError(f'Missing tokenized corpus file: {TOKENIZED_CORPUS_FILE}')
    if not BM25_INDEX_FILE.exists():
        raise FileNotFoundError(f'Missing BM25 index file: {BM25_INDEX_FILE}')

    document_metadata = load_document_metadata()
    tokenized_corpus = load_tokenized_corpus()
    _ = load_bm25_index()

    if len(document_metadata) != len(tokenized_corpus):
        raise ValueError('Document metadata row count does not match tokenized corpus length.')

    expected_doc_ids = np.arange(len(document_metadata), dtype=np.int64)
    actual_doc_ids = document_metadata['doc_id'].to_numpy(dtype=np.int64, copy=False)
    if not np.array_equal(actual_doc_ids, expected_doc_ids):
        raise ValueError('doc_id values are not dense and sorted from 0 to N-1.')

    if not 20_000 <= len(document_metadata) <= 22_500:
        print(
            f'Warning: expected about 21,000 documents, but found {len(document_metadata):,}. '
            'This is not a hard failure.'
        )
    else:
        print(f'Document count is in the expected range: {len(document_metadata):,}')

    validation_queries = [
        'small dogs good with children',
        'domestic cat',
        'golden retriever',
        'aquarium fish',
        'horse breed',
    ]

    validation_results = []
    for query in validation_queries:
        results = bm25_search(query, top_k=5, include_document_text=False)
        if results.empty:
            raise ValueError(f'Validation search returned no results for query: {query}')
        validation_results.append(
            {
                'query': query,
                'top_title': results.iloc[0]['title'],
                'top_score': float(results.iloc[0]['bm25_score']),
            }
        )

    empty_token_documents = sum(1 for tokens in tokenized_corpus if not tokens)

    print('Validation summary')
    print(f'- document metadata file exists: {DOCUMENT_METADATA_FILE.exists()}')
    print(f'- tokenized corpus file exists: {TOKENIZED_CORPUS_FILE.exists()}')
    print(f'- BM25 index file exists: {BM25_INDEX_FILE.exists()}')
    print(f'- metadata rows: {len(document_metadata):,}')
    print(f'- tokenized documents: {len(tokenized_corpus):,}')
    print(f'- empty-token documents: {empty_token_documents}')
    for item in validation_results:
        print(
            f"- query '{item['query']}' -> top result: {item['top_title']} "
            f"(score={item['top_score']:.4f})"
        )
    print('Validation passed.')

    return {
        'document_count': len(document_metadata),
        'empty_token_documents': empty_token_documents,
        'validation_queries': validation_results,
    }

## Hybrid Retrieval Stub

This BM25 notebook stays purely lexical, but it is designed to fit into a later hybrid retriever.

Intended future design:

- BM25 returns document-level lexical ranking.
- MiniLM returns chunk-level semantic scores aggregated to document-level ranking.
- Combine both rankings with Reciprocal Rank Fusion.

Recommended fusion formula:

`RRF(d) = 1 / (k + rank_bm25(d)) + 1 / (k + rank_semantic(d))`

Recommended settings:

- `k = 60`
- BM25 candidate depth: top 100
- MiniLM candidate depth: top 100
- final reranked top 10

## Practical Notes

- BM25 indexes full documents, not chunks.
- This is appropriate because BM25 does not have a transformer input-length limitation.
- BM25 works especially well for exact keywords, titles, rare names, scientific names, and specific terminology.
- BM25 may miss semantic paraphrases, which is why it should later be paired with MiniLM.
- Keep BM25 artifacts in `data/bm25_index` and embedding artifacts in `data/embeddings`.
- If long documents appear under-ranked, experiment with reducing `BM25_B` from `0.75` to `0.5`.
- If scores seem too term-frequency-heavy, experiment with `BM25_K1` values around `1.2` to `1.5`.

## Recommended Execution Order

In [8]:
build_document_metadata(overwrite=False)

Processed 500/21,077 page index entries in 7.3s | documents: 500 | skipped: 0
Processed 1,000/21,077 page index entries in 12.3s | documents: 999 | skipped: 1
Processed 1,500/21,077 page index entries in 17.4s | documents: 1,499 | skipped: 1
Processed 2,000/21,077 page index entries in 22.9s | documents: 1,998 | skipped: 2
Processed 2,500/21,077 page index entries in 27.2s | documents: 2,498 | skipped: 2
Processed 3,000/21,077 page index entries in 31.4s | documents: 2,998 | skipped: 2
Processed 3,500/21,077 page index entries in 35.3s | documents: 3,498 | skipped: 2
Processed 4,000/21,077 page index entries in 39.4s | documents: 3,997 | skipped: 3
Processed 4,500/21,077 page index entries in 42.9s | documents: 4,497 | skipped: 3
Processed 5,000/21,077 page index entries in 46.8s | documents: 4,997 | skipped: 3
Processed 5,500/21,077 page index entries in 52.6s | documents: 5,497 | skipped: 3
Processed 6,000/21,077 page index entries in 57.9s | documents: 5,997 | skipped: 3
Processed 6

,doc_id,page_id,title,file_name,text_file_original,word_count,char_count,document_text
0,0,7590101,(Blooper) Bunny,(Blooper)_Bunny_8dbe4cbf9c.txt,texts/(Blooper)_Bunny_8dbe4cbf9c.txt,1789,10424,(Blooper) Bunny is a Merrie Melodies animated ...
1,1,46645702,0.0. Duck,0.0._Duck_c54e2adeaa.txt,texts/0.0._Duck_c54e2adeaa.txt,20189,117302,The following Disney cartoon and comics charac...
2,2,72360454,10 Little Rubber Ducks,10_Little_Rubber_Ducks_1640dfcf6d.txt,texts/10_Little_Rubber_Ducks_1640dfcf6d.txt,117,732,10 Little Rubber Ducks is a 2005 children's bo...
3,3,58430204,10 Lives,10_Lives_be15f63f40.txt,texts/10_Lives_be15f63f40.txt,860,5007,"For other uses of 10 Lives or Ten Lives, see 1..."
4,4,34430924,10 Promises to My Dog,10_Promises_to_My_Dog_79e1c7b036.txt,texts/10_Promises_to_My_Dog_79e1c7b036.txt,1015,5782,"10 Promises To My Dog ( 犬と私の１０の約束 , Inu to Wat..."
...,...,...,...,...,...,...,...,...
21053,21053,72384500,ラウドボーン,ラウドボーン_9a864bedb8.txt,texts/ラウドボーン_9a864bedb8.txt,391,2356,"""Flittle"" redirects here. For the Disney chara..."
21054,21054,72384566,リキキリン,リキキリン_4857f53617.txt,texts/リキキリン_4857f53617.txt,391,2356,"""Flittle"" redirects here. For the Disney chara..."
21055,21055,72384803,リククラゲ,リククラゲ_2c97e8ac5f.txt,texts/リククラゲ_2c97e8ac5f.txt,391,2356,"""Flittle"" redirects here. For the Disney chara..."
21056,21056,72384616,ワッカネズミ,ワッカネズミ_78df9df5a7.txt,texts/ワッカネズミ_78df9df5a7.txt,391,2356,"""Flittle"" redirects here. For the Disney chara..."


In [9]:
build_tokenized_corpus(overwrite=False)

Tokenizing documents: 100%|██████████| 21058/21058 [00:27<00:00, 763.28it/s] 


Tokenized corpus statistics
- documents: 21,058
- total tokens: 15,917,618
- average document token count: 755.89
- median document token count: 184.50
- min document token count: 3
- max document token count: 25964
- empty-token documents: 0
Saved tokenized corpus to D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\bm25_index\tokenized_corpus.pkl


[['blooper',
  'bunny',
  'merrie',
  'melodies',
  'animated',
  'short',
  'film',
  'directed',
  'greg',
  'ford',
  'terry',
  'lennon',
  'music',
  'george',
  'daugherty',
  'produced',
  '1991',
  'warner',
  'bros',
  'animation',
  'featuring',
  'voice',
  'talents',
  'jeff',
  'bergman',
  'gordon',
  'hunt',
  'russell',
  'calabrese',
  'short',
  'parody',
  'specials',
  'produced',
  'bugs',
  'bunny',
  '50',
  'th',
  'anniversary',
  'previous',
  'year',
  'short',
  'never',
  'received',
  'intended',
  'theatrical',
  'release',
  'shelved',
  'six',
  'years',
  'finally',
  'given',
  'television',
  'premiere',
  'cartoon',
  'network',
  'june',
  '13',
  '1997',
  'featured',
  'looney',
  'tunes',
  'golden',
  'collection',
  'volume',
  '2003',
  'update',
  'bugs',
  'bunny',
  '80',
  'th',
  'anniversary',
  'collection',
  '2020',
  'update',
  'cartoon',
  'opens',
  'short',
  'special',
  'celebrating',
  'bugs',
  'bunnys',
  '51',
  'st',
  'h

In [10]:
build_bm25_index(overwrite=False)

Saved BM25 index to D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\bm25_index\bm25_index.pkl
Saved BM25 config to D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\bm25_index\bm25_config.json
Built BM25 index in 7.7s


In [13]:
validate_bm25_pipeline()

Document count is in the expected range: 21,058
Validation summary
- document metadata file exists: True
- tokenized corpus file exists: True
- BM25 index file exists: True
- metadata rows: 21,058
- tokenized documents: 21,058
- empty-token documents: 0
- query 'small dogs good with children' -> top result: Fu Quan (score=14.0566)
- query 'domestic cat' -> top result: Savannah cat (score=11.1213)
- query 'golden retriever' -> top result: Golden Retriever (score=17.2583)
- query 'aquarium fish' -> top result: Category:Fishkeeping (score=9.0923)
- query 'horse breed' -> top result: American Hackney Horse Society (score=12.0886)
Validation passed.


{'document_count': 21058,
 'empty_token_documents': 0,
 'validation_queries': [{'query': 'small dogs good with children',
   'top_title': 'Fu Quan',
   'top_score': 14.056571960449219},
  {'query': 'domestic cat',
   'top_title': 'Savannah cat',
   'top_score': 11.121336936950684},
  {'query': 'golden retriever',
   'top_title': 'Golden Retriever',
   'top_score': 17.258333206176758},
  {'query': 'aquarium fish',
   'top_title': 'Category:Fishkeeping',
   'top_score': 9.092296600341797},
  {'query': 'horse breed',
   'top_title': 'American Hackney Horse Society',
   'top_score': 12.08858871459961}]}

In [14]:
DEMO_QUERIES = [
    'small dogs good with children',
    'golden retriever temperament',
    'domestic cat behavior',
    'aquarium fish care',
    'working dog breed',
]

demo_frames = []
for query in DEMO_QUERIES:
    print(f'\nQuery: {query}')
    results = bm25_search(query, top_k=5, include_document_text=False)
    demo_view = results[['rank', 'bm25_score', 'title', 'page_id', 'file_name', 'matched_query_tokens']]
    print(demo_view.to_string(index=False))
    if not demo_view.empty:
        demo_frames.append(demo_view.assign(query=query))

if demo_frames:
    search_demo_results = pd.concat(demo_frames, ignore_index=True)
    search_demo_results.to_parquet(SEARCH_DEMO_RESULTS_FILE, index=False)
    print(f'\nSaved demo search results to {SEARCH_DEMO_RESULTS_FILE}')
else:
    print('\nNo demo search results were saved because all demo searches returned empty results.')


Query: small dogs good with children
 rank  bm25_score                    title  page_id                               file_name        matched_query_tokens
    1   14.056572                  Fu Quan 75208788                  Fu_Quan_7de609811a.txt small, dogs, good, children
    2   13.358662 Treeing Walker Coonhound  2370241 Treeing_Walker_Coonhound_df36d6b8a5.txt small, dogs, good, children
    3   13.225189     Pet culture in Japan 38859693     Pet_culture_in_Japan_a4ad864558.txt small, dogs, good, children
    4   13.192688        Bohemian Shepherd  9687545        Bohemian_Shepherd_ecff4ab79e.txt small, dogs, good, children
    5   12.966438          Norfolk Terrier  1580593          Norfolk_Terrier_dfc902ced3.txt small, dogs, good, children

Query: golden retriever temperament
 rank  bm25_score                  title  page_id                             file_name           matched_query_tokens
    1   19.839113           Goldendoodle   482709           Goldendoodle_3590eff194.tx